<div style="text-align:center;">
  <img src="https://github.com/LinkedEarth/Logos/blob/master/PaleoPAL/PaleoPal_rectangular_light.png?raw=true" width="500">
</div>

## PaleoPAL Evaluation: Notebook 3

This notebook is part of a series of evaluation tests for the [PaleoPAL](linked.earth/paleopal) assistant. You have two hours to complete the assignment. 

The notebook is divided into the following sections:
1. Data Gathering (1 hour and 15min)
2. Analysis (35 min)
3. Visualization (10min)

If you cannot complete the assignment for each section in the time alloted, use the solution and move on to the next section. 

**Use VS Code to complete the assignment**. 

In [17]:
### Import libraries

## Motivation

The **²³¹Pa/²³⁰Th ratio** is a paleoclimate proxy that tracks past changes in Atlantic Meridional Overturning Circulation (AMOC). When AMOC is strong, ²³⁰Th is preferentially scavenged from the water column, raising the Pa/Th ratio above the production ratio of ~0.093. A ratio close to or below 0.093 indicates weak or absent overturning. This makes Pa/Th one of the most direct tracers of past ocean circulation changes.

## Data Gathering (1hr 15min)

In this part of the assignment, you will gather data from three different sources: PANGAEA through PyleoTUPS, a local LiPD file via PyLiPD, and a SPAQRL endpoint to the LiPDGraph.

### PyleoTUPS

You will use the data from [Ng et al. (2018)](https://doi.org/10.1038/s41467-018-05312-3), which can be accessed [here](https://doi.pangaea.de/10.1594/PANGAEA.890942).

**Asignment task:**

a. Search for the study using the information from the [PANGAEA page](https://doi.pangaea.de/10.1594/PANGAEA.890942). For this work, we will be using the ²³¹Pa/²³⁰Th ratio from core EW9209-3JPC.

b. Display the study summary so you can verify the citation and dataset identity.

c. Display the geographic information so you can recover the site name, latitude, and longitude.

d. Load the data table and inspect the first rows. Make note of the column names and how they can be used later.


In [16]:
dsp = pt.PangaeaDataset()


res = dsp.search_studies(study_ids='890928')
display(res)

df_geo = dsp.get_geo()
display(df_geo)

df_data = dsp.get_data(study_id = '890928')[0]
display(df_data.head())

### PyLiPD

Load the data in the `Ng.EW9209-1JPC.2018.lpd` and retrieve information about the timeseries contain in the file. 

**Assignment task:**
a. Open the dataset and retrieve relevant timeseries information
b. Filter the dataframe to keep only rows where the paleo variable is ²³¹Pa/²³⁰Th ratio.

In [18]:
lipd = LiPD()
lipd.load("Ng.EW9209-1JPC.2018.lpd")

df = lipd.get_timeseries_essentials(lipd.get_all_dataset_names())

df

### LiPDGraph 

Using a SPARQL query on the LiPDGraph, search for ²³¹Pa/²³⁰Th records of marine sediments.

**Assignment task:**

a. Query the LiPDGraph endpoint (https://linkedearth.graphdb.mint.isi.edu/repositories/LiPDVerse-dynamic) for all relevant records of `231Pa/230Th` on `Marine sediment`. Retrieve all timeseries relevant information (e.g., time name, values, and units; paleo variable name, values and units; archive type; geographical location...)
b. Convert the query response into a `pandas.DataFrame`.
c. Remove duplicate rows from the resulting DataFrame 

In [14]:
query = """

PREFIX le: <http://linked.earth/ontology#>
PREFIX le_var: <http://linked.earth/ontology/paleo_variables#>
PREFIX wgs84: <http://www.w3.org/2003/01/geo/wgs84_pos#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?dataSetName ?archiveType ?geo_meanLat ?geo_meanLon
    ?paleoData_variableName  ?paleoData_values ?paleoData_units 
    ?time_variableName ?time_values ?time_units
WHERE {
    ?ds a le:Dataset .
    ?ds le:hasName ?dataSetName .

    ?ds le:hasArchiveType ?atObj . ?atObj rdfs:label ?archiveType .
    FILTER(regex(?archiveType, 'Marine sediment.*',"i"))


    ?ds le:hasLocation ?loc .
    OPTIONAL { ?loc wgs84:lat  ?geo_meanLat . }
    OPTIONAL { ?loc wgs84:long ?geo_meanLon . }

    ?ds le:hasPaleoData ?data .
    ?data le:hasMeasurementTable ?table .

    ?table le:hasVariable ?var .
    ?var le:hasName ?paleoData_variableName .
    FILTER(regex(?paleoData_variableName, '.*P/.*Th.*',"i"))
    ?var le:hasValues ?paleoData_values .
    OPTIONAL { ?var le:hasUnits ?uObj . ?uObj rdfs:label ?paleoData_units . }

    ?table le:hasVariable ?timevar .
    ?timevar le:hasName ?time_variableName .
    ?timevar le:hasValues ?time_values .
    OPTIONAL { ?timevar le:hasUnits ?tuObj . ?tuObj rdfs:label ?time_units . }
    ?timevar le:hasStandardVariable le_var:age .
}
limit 10
""" 

url = 'https://linkedearth.graphdb.mint.isi.edu/repositories/LiPDVerse-dynamic'

response = requests.post(url, data = {'query': query})

data = io.StringIO(response.text)
df_graph = pd.read_csv(data, sep=",")

df_graph.head()

## Analysis (35 min)

You will use Ammonyte to apply the  same changepoint detection method (ruptures) to each of the datasets assembled above.  

**Assignment Task:**
a. Wrap the Pa/Th and age data in an `ammonyte.Series`
b. Interpolate to an **evenly spaced** time grid
c. Run the ruptures method with the **same algorithm and penalty** for all three, ensuring a fair 

In [13]:
# Shared rupture settings — same across all three datasets for fair comparison
ALGO   = 'Pelt'
COST   = 'rbf'
PEN    = 3       # lower = more sensitive; adjust based on your signal
INTERP_STEP = 0.5  # interpolation step in years

# Create Series
ts_pangaea = amt.Series(time = df_data['Age'], 
                        value = df_data['231Pa/230Th xs0'],
                        label='EW9209-3JPC',
                        time_name='Age',
                        time_unit='yr BP',
                        value_name='231Pa/230Th',
                        value_unit='unitless')

ts_lipd = amt.Series(time = df['time_values'].iloc[0], 
                     value = df['paleoData_values'].iloc[0],
                     label='EW9209-1JPC',
                     time_name='Age',
                     time_unit='yr BP',
                     value_name='231Pa/230Th',
                     value_unit='unitless')

#Interpolate
ts_pangaea_interp = ts_pangaea.interp(step=INTERP_STEP)
ts_lipd_interp = ts_lipd.interp(step=INTERP_STEP)

#ruptures
transitions_pangaea = ts_pangaea_interp.ruptures(algo=ALGO, cost=COST,pen=PEN)
print(transitions_pangaea)

transitions_lipd = ts_lipd_interp.ruptures(algo=ALGO, cost=COST,pen=PEN)
print(transitions_lipd)

## Visualization (10 min)

**Assignment Task:**

a. Visualize the transitions

In [12]:
fig, ax = transitions_pangaea.plot()
plt.title('EW9209-3JPC')
plt.tight_layout()
plt.show()

fig, ax = transitions_lipd.plot()
plt.title('EW9209-1JPC')
plt.tight_layout()
plt.show()